### Code Attribution
Some parts of this notebook, specifically the hybrid recommendation scoring, were inspired by the Kaggle notebook by Nikolay Shivarov (2024): 
[Hybrid Music Recommendation System](https://www.kaggle.com/code/nikolayshivarov2000/hybrid-usic-recommendation-system/notebook). Each section of the code inspired by this, will be mentioned in a markdown above given cellbox.

Other parts of the code, including dataset preprocessing, kNN implementation, and normalization steps, are original.


# Overview

### Importing Datasets and Python Packages

In [145]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix,coo_matrix

In [146]:
music_info = pd.read_csv('Data/music_info_cleaned.csv')
users_history = pd.read_csv('Data/users_history_cleaned.csv')
music_info_metadata = pd.read_csv('Data/music_info_metadata.csv')

In [147]:
music_info.head()

,track_id,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,...,tag_synthpop,tag_techno,tag_thrash_metal,tag_trance,tag_trip_hop,artist_encoded,mood_Q1,mood_Q2,mood_Q3,mood_Q4
0,TRIOREW128F424EAF0,2004,222200,0.360041,0.918,1,0.874265,1.0,0.078197,0.001195,...,0,0,0,0,0,7209,0,1,0,0
1,TRRIVDJ128F429B0E8,2006,258613,0.414807,0.892,2,0.874061,1.0,0.035220,0.000810,...,0,0,0,0,0,5262,1,0,0,0
2,TROUVHL128F426C441,1991,218920,0.515213,0.826,4,0.851906,0.0,0.041929,0.000176,...,0,0,0,0,0,5194,1,0,0,0
3,TRUEIND128F93038C4,2004,237026,0.282961,0.664,9,0.803699,1.0,0.038889,0.000391,...,0,0,0,0,0,2672,0,0,0,1
4,TRLNZBD128F935E4D8,2008,238640,0.522312,0.430,7,0.786666,1.0,0.038679,0.010241,...,0,0,0,0,0,5740,0,0,1,0


In [148]:
users_history.head()

,track_id,user_id,playcount
0,TRLATHU128F92FC275,5a905f000fc1ff3df7ca807d57edb608863db05d,0.370370
1,TRMKFPN128F42858C3,5a905f000fc1ff3df7ca807d57edb608863db05d,0.037037
2,TRTSSUT128F1472A51,5a905f000fc1ff3df7ca807d57edb608863db05d,0.000000
3,TRNJLKP128F427CE28,5a905f000fc1ff3df7ca807d57edb608863db05d,0.000000
4,TRGAOLV128E0789D40,5a905f000fc1ff3df7ca807d57edb608863db05d,0.037037


In [149]:
music_info_metadata.head()

,track_id,name,artist
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis
2,TROUVHL128F426C441,Come as You Are,Nirvana
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand
4,TRLNZBD128F935E4D8,Creep,Radiohead


For better output display, we create a function that takes a track ID and looks up the track's name and artist from the metadata dictionary that we saved in the preprocessing notebook. If the track ID is not found, it will return a default dictionary with the track ID as the name and "Unknown" as the artist.

In [150]:
# Function to map a track ID to its metadata
def track_id_to_metadata(track_id, track_metadata_dict):
    return track_metadata_dictget(track_id, {"name": track_id, "artist": "Unknown"})

### Prepare Numerical Features for Content-Based Recommendations

In [151]:
# Features to be used to compute similarity between traks using kNN
feature_columns = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 
                   'instrumentalness', 'liveness', 'valence', 'tempo']
numerical_features = music_info[feature_columns].values

### k-Nearest Neighbors (kNN) for Content-Based Recommendations

In the original Kaggle notebook by Shivarov (2024) which we took inspiration from, the author uses an Annoy index to find similar tracks for the content-based part of the hybrid recommender. Annoy is an approximate nearest neighbors library used for very fast similarity searches on large datasets, which is useful for datasets containing millions of items. In our implementation, we chose to use k-Nearest Neighbors (kNN) from scikit-learn instead. kNN performs exact similarity searches, and is often simpler to implement. The main trade-off is between speed and exactness, where Annoy is faster but approximate, whereas kNN is exact but can be slower on large datasets. Since we are already more familiar with kKNN, we found it easier to use and sufficient for our dataset size.
> **Source**:
>  Shivarov, N. (2024). *Hybrid Music Recommendation System*. Kaggle Notebook. Retrieved from [link](https://www.kaggle.com/code/nikolayshivarov2000/hybrid-usic-recommendation-system/notebook) (Accessed: 14 October 2025)

In [152]:
from sklearn.neighbors import NearestNeighbors

# Fit kNN model for content-baesd recommendations
knn_index = NearestNeighbors(n_neighbors=50, algorithm='auto', metric='cosine').fit(numerical_features)


Given the track ID, called item_id here, the function below queries the kNN model from above to find the most similar tracks. We return the indices of the top n most similar tracks, excluding the track itself. We also return the corresponding distances, which can be used later as similarity scores.

In [153]:
# Function to get nearest neighbors for a given track
def get_knn(knn_index, feature_matrix, item_id, n):
    distances, indices = knn_index.kneighbors(numerical_features[item_id:item_id+1], n_neighbors=n+1)
    # Return so it skips the first one (itself)
    return indices[0][1:], distances[0][1:]


In [154]:
train, test = train_test_split(users_history, test_size=0.2, random_state=42)

These mappings below, inspired by Shivarov (2024) includes both forward (integer to original ID) and reverse (original ID to integer) dictionaries for easy lookup.

> **Note** The codebox below was inspired by:
>  Shivarov, N. (2024). *Hybrid Music Recommendation System*. Kaggle Notebook. Retrieved from [link](https://www.kaggle.com/code/nikolayshivarov2000/hybrid-usic-recommendation-system/notebook) (Accessed: 14 October 2025)

In [155]:
# For collaborative filtering (using users_history)
train['user_id'] = train['user_id'].astype('category')
train['track_id'] = train['track_id'].astype('category')

# Create mappings for user_id and track_id
cf_user_id_mapping = dict(enumerate(train['user_id'].cat.categories))
cf_track_id_mapping = dict(enumerate(train['track_id'].cat.categories))
cf_user_id_reverse_mapping = {v: k for k, v in cf_user_id_mapping.items()}
cf_track_id_reverse_mapping = {v: k for k, v in cf_track_id_mapping.items()}

# For content-based filtering (using music_info)
music_info['track_id'] = music_info['track_id'].astype('category')

cb_track_id_mapping = dict(enumerate(music_info['track_id'].cat.categories))
cb_track_id_reverse_mapping = {v: k for k, v in cb_track_id_mapping.items()}



The codebox below inspired by Shivarov (2024), converts user-track playcount data into a sparse matrix, and as a safety-measurement we fill missing playcounts with zeros. Singular Value Decomposition (SVD) decomposes this matrix into **user factors** and **item factors**.

> **Note** The codebox below was inspired by:
>  Shivarov, N. (2024). *Hybrid Music Recommendation System*. Kaggle Notebook. Retrieved from [link](https://www.kaggle.com/code/nikolayshivarov2000/hybrid-usic-recommendation-system/notebook) (Accessed: 14 October 2025)


In [156]:
from scipy.sparse import coo_matrix

user_item_sparse = coo_matrix((
    train['playcount'],
    (train['user_id'].cat.codes,
     train['track_id'].cat.codes)
))

# Apply SVD on the Sparse Matrix
svd = TruncatedSVD(n_components=10, random_state=42)
user_factors = svd.fit_transform(user_item_sparse)
item_factors = svd.components_.T

In [157]:
user_item_sparse.shape

(22733, 25903)

Hybrid recommendation system learnt during Seddik's lecture (2025) for music that combines: 
- **Collaborative Filtering (SVD-based)**: predicting user preferences based on latent factors from user-track interactions
- **Content-Based Filtering (kNN on audio features):** finding similar tracks given audio features
- **Mood scores**: for each song checks if mood of song matches users mood and assigns scores 1 for match and 0 for mismatch
- **Score normalization**: ensures CD-, CB- and mood scores are on comparable scales to prevent one from dominating the other.
- **Hyrbid:** combines CF-, CB- and mood scores with beta weights to balance their influence. Inspired by lecture by Seddik (2025) on Hybrid-Knowledge-Based recommender systems.
The result of all of the above is a ranked list that reflects both user's history, similarity to chosen track and current emotional state.

> **Source**: Seddik, K.M.A., 2025. *Hybrid-knowledge-based*. Lecture, University of Bergen, 27 October 2025.



Parameters:
- **user_id:** ID of the user to make recommendations for
- **track_name:** Name of the track to use as reference for recommendations
- **user_item_matrix:** User-item interaction matrix
- **user_factors:** Latent user factors from SVD (CF)
- **item_factors:** Latent item factors from SVD (CF)
- **music_info_metadata:** Track metadata (name, artist, etc.)
- **knn_index:** kNN index for CB
- **numerical_features:** Numerical audio features for CB similarity
- **user_mood:** Asking user for mood input, which is set to None if not provided
- **n_recommendations:** Number of hybrid recommendations to return



> **Note** The codebox below was inspired by:
>  Shivarov, N. (2024). *Hybrid Music Recommendation System*. Kaggle Notebook. Retrieved from [link](https://www.kaggle.com/code/nikolayshivarov2000/hybrid-usic-recommendation-system/notebook) (Accessed: 14 October 2025)
>
> Normalization, track metadata, validating track, scores and weighted hybridization are original

In [158]:
def recommend_songs_hybrid(user_id, track_name, user_item_matrix, user_factors, item_factors,
                           music_info_metadata, knn_index, numerical_features, user_mood=None,
                           n_recommendations=5):

    # Using metadata from earlier containing artist and name of track
    track_metadata_dict = music_info_metadata.set_index('track_id')[['name', 'artist']].to_dict('index')
    
    # Check if user_id exists in mapping, code by Shivarov (2024)
    user_code = cf_user_id_reverse_mapping.get(user_id)
    if user_code is None:
        print(f"User ID {user_id} not found in the user-item matrix.")
        return []

    # Validate track, inspired by Shivarov (2024)
    track_row = music_info_metadata[music_info_metadata['name'] == track_name]
    if track_row.empty:
        print(f"Track '{track_name}' not found in metadata.")
        return []
        
    track_id = track_row.iloc[0]['track_id']
    # Track code used for CF scores
    track_code = cb_track_id_reverse_mapping.get(track_id)
    if track_code is None:
        print(f"Track '{track_name}' not found.")
        return [], []

    # -------------------
    # Content-Based Filtering with kNN
    # -------------------
    similar_indices, distances_from_knn = get_knn(knn_index, numerical_features, track_code, n=25)
    cb_candidates = music_info_metadata.iloc[similar_indices][['track_id', 'name', 'artist']].copy()
    cb_recommended_tracks = cb_candidates['track_id'].tolist()

    # Converting kNN distances into similarity scores (closer distance = higher score)
    x = 1 - distances_from_knn 
    cb_scores = dict(zip(cb_recommended_tracks, x))

    # -------------------
    # Collaborative Filtering 
    # -------------------
    cf_predictions = np.dot(user_factors[user_code, :], item_factors.T)
    cf_indices = np.argsort(cf_predictions)[::-1]
    cf_recommended_tracks = [cf_track_id_mapping[i] for i in cf_indices[:n_recommendations]]
    
    # CF predicted scores
    cf_scores = {}
    for track_id in cf_recommended_tracks:
        # Dot product between user and item factors is the predicted preference
        score = np.dot(user_factors[user_code], item_factors[track_code])
        cf_scores[track_id] = score


    # Normalizating score to ensure one does not dominate the other
    print("CF score range BEFORE normalization:", min(cf_scores.values()), max(cf_scores.values()))
    print("CB score range BEFORE normalization:", min(cb_scores.values()), max(cb_scores.values()))
    
    def min_max_normalize(scores):
        values = np.array(list(scores.values()))
        min_val = values.min()
        max_val = values.max()
        
        # If all values are the same, avoid division by zero
        if max_val == min_val:
            return {item_id: 0.5 for item_id in scores} 
    
        # Apply min–max normalization to each score
        return {
            item_id: (value - min_val) / (max_val - min_val)
            for item_id, value in scores.items()
        }

        
    # Normalize CF and CB scores
    cf_scores_norm = min_max_normalize(cf_scores)
    cb_scores_norm = min_max_normalize(cb_scores)

    print("CF score range AFTER normalization:", min(cf_scores_norm.values()), max(cf_scores_norm.values()))
    print("CB score range AFTER normalization:", min(cb_scores_norm.values()), max(cb_scores_norm.values()))

    # -------------------
    # Mood-based scoring
    # -------------------

    mood_scores = {}

    # Combinging candidates for suggestions from CF and CB
    candidate_tracks = set(cf_scores_norm) | set(cb_scores_norm)
    mood_column = f'mood_{user_mood}'

    for track_id in candidate_tracks:
        # Get mood value for track (1 if match, 0 otherwise)
        mood_value = music_info.loc[music_info['track_id'] == track_id, mood_column].values[0]
        mood_scores[track_id] = mood_value


    # -------------------
    # Hybrid Recommendations
    # -------------------
    
    # Weighted combination (beta_cf + beta_cb + beta_mood = 1)
    beta_cf = 0.2
    beta_cb = 0.2
    beta_mood = 0.6
    
    hybrid_scores = {}

    for track_id in candidate_tracks:
        # Calculate each weighted score for the given song
        cf_score = cf_scores_norm.get(track_id, 0)
        cb_score = cb_scores_norm.get(track_id, 0)
        mood_score = mood_scores.get(track_id, 0)
        
        hybrid_scores[track_id] = (
            beta_cf * cf_score +
            beta_cb * cb_score +
            beta_mood * mood_score
        )
        
    # Sorting by hybrid score in descending order
    hybrid_sorted = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)

    # Selecting top n recommendations
    hybrid_recommended_tracks = [track_id for track_id, score in hybrid_sorted[:n_recommendations]]

    
    # Display recommendations
    print("===========================================================\n")
    print(f"CF Recommendations for user {user_id}:")
    for t in cf_recommended_tracks:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    
    print("\n===========================================================\n")
    print(f"CB (kNN) Similar Tracks to '{track_name}':")
    for t in cb_recommended_tracks[:n_recommendations]:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    
    print("\n===========================================================\n")
    print(f"Hybrid Recommendations:")
    for t in hybrid_recommended_tracks:
        meta = track_metadata_dict.get(t, {"name": t, "artist": "Unknown"})
        print(f" - {meta['name']} by {meta['artist']}")
    print("===========================================================\n")

    return hybrid_recommended_tracks


#### Training

In [159]:
# Converting user interactions into dictionary for train/test
# Inspired by Shivarov (2024) Kaggle notebook.
user_train_data = train.groupby('user_id', observed=True)['track_id'].apply(list).to_dict()
user_test_data = test.groupby('user_id', observed=True)['track_id'].apply(list).to_dict()

#### Preparing and Running the Recommender

Based on the users mood input, we use the given category to recommend songs.

In [160]:
# Dictionary mapping every emotion term to its quadrant
emotion_to_quadrant = {
    # Q1 (High Valence, High Arousal)
    "elation": "Q1", 
    "amusement": "Q1", 
    "happiness": "Q1", 
    "surprise": "Q1",
    "joy": "Q1", 
    "cheerful": "Q1", 
    "love": "Q1", 
    "pleasure": "Q1", 
    "proud": "Q1",
    "pleased": "Q1", 
    "amazed": "Q1", 
    "excited": "Q1", 
    "delighted": "Q1", 
    "glad": "Q1",

    # Q2 (Low Valence, High Arousal)
    "panic fear": "Q2", 
    "hot anger": "Q2", 
    "despair": "Q2", 
    "anxiety/worry": "Q2",
    "anger": "Q2", 
    "fear": "Q2", 
    "disgust": "Q2", 
    "nervous": "Q2", 
    "scary": "Q2",
    "anxiety": "Q2", 
    "irritation": "Q2", 
    "hate": "Q2", 
    "frustrated": "Q2",
    "worried": "Q2", 
    "furious": "Q2", 
    "alarmed": "Q2", 
    "annoyed": "Q2",
    "distressed": "Q2",

    # Q3 (Low Valence, Low Arousal)
    "interest": "Q3", 
    "sadness": "Q3", 
    "cold anger/irritation": "Q3", 
    "worried": "Q3",
    "bored": "Q3", 
    "despair": "Q3", 
    "disgust": "Q3", 
    "contempt": "Q3",
    "disappointment": "Q3", 
    "threatened": "Q3", 
    "afraid": "Q3", 
    "tender": "Q3",
    "tired": "Q3", 
    "depressed": "Q3", 
    "distressed": "Q3",

    # Q4 (High Valence, Low Arousal)
    "pride": "Q4", 
    "relief": "Q4", 
    "sensual pleasure": "Q4", 
    "tenderness": "Q4",
    "calm": "Q4", 
    "satisfied": "Q4", 
    "sadness": "Q4", 
    "peacefulness": "Q4",
    "contentment": "Q4", 
    "interest": "Q4", 
    "compassion": "Q4", 
    "loving": "Q4",
    "comforted": "Q4", 
    "relaxed": "Q4", 
    "melancholy": "Q4"
}

# Create list of all valid mood strings for user prompt
valid_emotions = list(emotion_to_quadrant.keys())


def ask_user_for_emotion():
    print("Available moods:\n")
    for mood in sorted(valid_emotions):
        print("-", mood)
    print()

    user_input = input("How are you feeling? Type one of the moods listed above:\n").strip().lower()

    # validating input
    while user_input not in emotion_to_quadrant:
        user_input = input("Invalid mood. Please type one of the moods exactly as listed:\n").strip().lower()

    quadrant = emotion_to_quadrant[user_input]
    print(f"You selected: {user_input.capitalize()} → categorized as {quadrant}")

    return quadrant


Since we are already using cosine similarity on vectors, each song has mood encoded into vectors as well. For the similarity to work, the user vector above must also be in the same feature space since the string is not compatible with cosine similarity.

In [161]:
# One-hot encode Q1, Q2, Q3, Q4 for user
def mood_quadrant_to_vector(quadrant):
    q_map = {"Q1": [1,0,0,0], "Q2": [0,1,0,0], "Q3": [0,0,1,0], "Q4": [0,0,0,1]}
    return q_map[quadrant]


In [162]:
import random

# Find a random user from user history to test recommendations on
random_row = users_history.sample(n=1)
random_user_id = random_row['user_id'].values[0]
print(random_user_id)


15d45e2a0aea0baf0678fc6ada6dc83c5ef04b2a


In [163]:
# Get the tracks this user has listened to in the training data
user_tracks = train[train['user_id'] == random_user_id]['track_id'].tolist()

if not user_tracks:
    print(f"No listening history found for user {random_user_id}.")
else:
    # Pick a random track from their listening history
    user_track_id = random.choice(user_tracks)
    # Retrieve track name and artist from metadata
    track_info = music_info_metadata.loc[
        music_info_metadata['track_id'] == user_track_id, ['name', 'artist']
    ].iloc[0]
    
    random_track_name = track_info['name']
    random_track_artist = track_info['artist']

    print(f"User {random_user_id} listened to: {random_track_name} by {random_track_artist}")


User 15d45e2a0aea0baf0678fc6ada6dc83c5ef04b2a listened to: We Like Sportz by The Lonely Island


In [164]:
recommend_songs_hybrid(
    random_user_id,
    random_track_name,
    user_item_sparse,
    user_factors,
    item_factors,
    music_info_metadata,
    knn_index,
    numerical_features,
    user_mood = ask_user_for_emotion(),
    n_recommendations=5
)

Available moods:

- afraid
- alarmed
- amazed
- amusement
- anger
- annoyed
- anxiety
- anxiety/worry
- bored
- calm
- cheerful
- cold anger/irritation
- comforted
- compassion
- contempt
- contentment
- delighted
- depressed
- despair
- disappointment
- disgust
- distressed
- elation
- excited
- fear
- frustrated
- furious
- glad
- happiness
- hate
- hot anger
- interest
- irritation
- joy
- love
- loving
- melancholy
- nervous
- panic fear
- peacefulness
- pleased
- pleasure
- pride
- proud
- relaxed
- relief
- sadness
- satisfied
- scary
- sensual pleasure
- surprise
- tender
- tenderness
- threatened
- tired
- worried



How are you feeling? Type one of the moods listed above:
 pride


You selected: Pride → categorized as Q4
CF score range BEFORE normalization: 0.0006637855024499614 0.0006637855024499614
CB score range BEFORE normalization: 0.9951669780739845 0.9980339225858521
CF score range AFTER normalization: 0.5 0.5
CB score range AFTER normalization: 0.0 1.0

CF Recommendations for user 15d45e2a0aea0baf0678fc6ada6dc83c5ef04b2a:
 - Sun It Rises by Fleet Foxes
 - White Winter Hymnal by Fleet Foxes
 - Your Protector by Fleet Foxes
 - Tiger Mountain Peasant Song by Fleet Foxes
 - He Doesn't Know Why by Fleet Foxes


CB (kNN) Similar Tracks to 'We Like Sportz':
 - Such Small Hands by La Dispute
 - Futures by Zero 7
 - Map Of Your Head by Muse
 - Believe in Us by Jay-Jay Johanson
 - Cheryl Tweedy by Lily Allen


Hybrid Recommendations:
 - White Winter Hymnal by Fleet Foxes
 - What's Up? by 4 Non Blondes
 - The Good Stuff by Kenny Chesney
 - Waiting... by City and Colour
 - Who Are You by The Who



['TRUJOHU128F424E6A6',
 'TRVHVJV128F426976F',
 'TRAMTZM12903D01F9E',
 'TRVIJEM128F42831B7',
 'TRUFMFC12903CBEFB7']